# Project 15 — Local Contract Clause Finder
## Retrieve Risky or Important Clauses from Contracts

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Step 2 — Sample Contract Content

In [ ]:
from langchain.schema import Document

contract_clauses = [
    Document(page_content="""LIMITATION OF LIABILITY: In no event shall either party
be liable for any indirect, incidental, special, consequential, or punitive
damages, regardless of the cause of action. The total cumulative liability
shall not exceed the fees paid in the 12 months preceding the claim.""",
        metadata={"clause_type": "liability", "section": "8.1", "risk": "high"}),

    Document(page_content="""TERMINATION FOR CONVENIENCE: Either party may terminate
this agreement for any reason with 30 days written notice. Upon termination,
Customer shall pay all fees for services rendered through the termination date.""",
        metadata={"clause_type": "termination", "section": "9.2", "risk": "medium"}),

    Document(page_content="""NON-COMPETE: During the term and for 24 months after
termination, neither party shall directly solicit employees of the other party.
This restriction applies globally with no geographic limitation.""",
        metadata={"clause_type": "non-compete", "section": "11.1", "risk": "high"}),

    Document(page_content="""DATA PROCESSING: Provider shall process personal data
only as instructed by Customer. Provider implements AES-256 encryption at rest
and TLS 1.3 in transit. Data breach notification within 72 hours.""",
        metadata={"clause_type": "data_protection", "section": "7.3", "risk": "medium"}),

    Document(page_content="""INTELLECTUAL PROPERTY: All IP created during the
engagement belongs to the Customer. Provider retains rights to pre-existing IP
and general knowledge. Provider grants Customer a perpetual license to
pre-existing IP incorporated into deliverables.""",
        metadata={"clause_type": "ip", "section": "6.1", "risk": "high"}),

    Document(page_content="""PAYMENT TERMS: Invoices are due within 30 days of receipt.
Late payments accrue interest at 1.5% per month or the maximum legal rate.
Customer may dispute invoices within 15 days of receipt.""",
        metadata={"clause_type": "payment", "section": "4.2", "risk": "low"}),
]
print(f"Loaded {len(contract_clauses)} contract clauses")

## Step 3 — Build Contract Index

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    contract_clauses, embeddings,
    persist_directory="sample_data/contract_chroma",
    collection_name="contracts",
)
print("Contract index ready!")

## Step 4 — Risk-Aware Clause Search

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

clause_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a contract review assistant. Analyze the contract clauses
and answer the question. For each relevant clause:
1. Quote the key language
2. Assess risk level (low/medium/high)
3. Flag any concerning terms
4. Suggest modifications to better protect the client

Contract clauses:
{context}

Question: {question}

Analysis:"""
)

qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": clause_prompt},
)

queries = [
    "What are the liability limitations in this contract?",
    "Is there a non-compete clause and what are its terms?",
    "How is intellectual property handled?",
    "What are the data protection obligations?",
]

for q in queries:
    print(f"\n{'='*60}\nQ: {q}")
    result = qa.invoke({"query": q})
    print(f"A: {result['result']}")

## Step 5 — Risk Summary Report

In [ ]:
from pydantic import BaseModel, Field

class ClauseRisk(BaseModel):
    clause_type: str
    section: str
    risk_level: str
    concern: str
    recommendation: str

class ContractRiskReport(BaseModel):
    overall_risk: str = Field(description="low/medium/high/critical")
    high_risk_clauses: list[ClauseRisk]
    recommendations: list[str]

risk_llm = llm.with_structured_output(ContractRiskReport)

all_text = "\n\n".join([c.page_content for c in contract_clauses])
report = risk_llm.invoke(f"Generate a risk report for these contract clauses:\n\n{all_text}")

print(f"Overall Risk: {report.overall_risk}")
print(f"\nHigh-Risk Clauses:")
for c in report.high_risk_clauses:
    print(f"  Section {c.section} ({c.clause_type}): {c.concern}")
    print(f"    → {c.recommendation}")
print(f"\nTop Recommendations:")
for r in report.recommendations:
    print(f"  • {r}")

## What You Learned
- **Legal clause indexing** with risk metadata
- **Risk-aware retrieval** and clause analysis
- **Structured risk reports** with actionable recommendations